# Research Abstract Classification — Exploration & Model Selection

**Goal:** classify research abstracts into Science, Business and Social Science.

This notebook follows the supplied project brief: text preprocessing, exploratory analysis, six classifier comparisons, evaluation metrics and confusion matrices. It expects `data/abstracts_labeled.csv` in the project folder. In Google Colab, upload the CSV or mount the project folder and update `DATA_PATH`.

In [ ]:
# Install dependencies in Colab (safe to rerun)
!pip -q install pandas numpy scikit-learn nltk matplotlib seaborn

In [ ]:
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
warnings.filterwarnings("ignore")
RANDOM_STATE=42
DATA_PATH=Path("data/abstracts_labeled.csv")
if not DATA_PATH.exists():
    from google.colab import files
    uploaded=files.upload()
    DATA_PATH=Path(next(iter(uploaded)))
df=pd.read_csv(DATA_PATH)
print("Shape:",df.shape)
display(df.head())

## 1. Data quality and cleaning

In [ ]:
print("Columns:", df.columns.tolist())
print("\\nMissing values (top):")
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing"))
print("\\nLabel counts:")
display(df["label"].value_counts().to_frame("samples"))
print("Duplicate abstracts:",df["Abstract"].fillna("").str.casefold().duplicated().sum())
print("Empty abstracts:",df["Abstract"].fillna("").str.strip().eq("").sum())

# Retain only usable rows and remove duplicate abstracts.
df["Abstract"]=df["Abstract"].fillna("").astype(str).str.strip()
df=df[df["Abstract"].ne("")].copy()
# Remove any text associated with conflicting labels, then exact duplicate abstracts.
conflicts=df.groupby(df["Abstract"].str.casefold())["label"].nunique()
ambiguous=set(conflicts[conflicts>1].index)
df=df[~df["Abstract"].str.casefold().isin(ambiguous)]
df=df.drop_duplicates(subset=["Abstract"]).reset_index(drop=True)
print("Cleaned shape:",df.shape," | ambiguous texts removed:",len(ambiguous))
display(df["label"].value_counts().to_frame("samples"))

## 2. Exploratory data analysis

In [ ]:
df["abstract_chars"]=df["Abstract"].str.len()
df["abstract_words"]=df["Abstract"].str.split().str.len()
fig,axes=plt.subplots(1,2,figsize=(13,4))
sns.countplot(data=df,x="label",order=df.label.value_counts().index,ax=axes[0])
axes[0].set_title("Abstracts per class"); axes[0].tick_params(axis="x",rotation=20)
sns.boxplot(data=df,x="label",y="abstract_words",ax=axes[1])
axes[1].set_title("Abstract word-count distribution"); axes[1].tick_params(axis="x",rotation=20)
plt.tight_layout(); plt.show()
display(df.groupby("label")[["abstract_chars","abstract_words"]].describe().round(1))

## 3. NLTK text preprocessing and TF-IDF

In [ ]:
import re
from nltk.tokenize import wordpunct_tokenize
def clean_text(text):
    text=re.sub(r"https?://\\S+|www\\.\\S+"," ",str(text).lower())
    text=re.sub(r"[^a-zA-Z\\s'-]"," ",text)
    tokens=wordpunct_tokenize(text)
    try:
        from nltk.corpus import stopwords
        stops=set(stopwords.words("english"))
    except LookupError:
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        stops=set(ENGLISH_STOP_WORDS)
    stops -= {"no","nor","not","against","without"}
    return " ".join(t for t in tokens if len(t)>1 and t not in stops)
print("BEFORE:",df.Abstract.iloc[0][:300])
print("AFTER: ",clean_text(df.Abstract.iloc[0])[:300])

## 4. Train/test split and model selection

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    df["Abstract"],df["label"],test_size=0.20,random_state=RANDOM_STATE,stratify=df["label"])
classifiers={
 "Logistic Regression":LogisticRegression(max_iter=2000,class_weight="balanced"),
 "Decision Tree":DecisionTreeClassifier(max_depth=80,min_samples_leaf=2,class_weight="balanced",random_state=RANDOM_STATE),
 "Naive Bayes":MultinomialNB(alpha=0.5),
 "kNN":KNeighborsClassifier(n_neighbors=5,weights="distance",metric="cosine",algorithm="brute"),
 "SVM":LinearSVC(class_weight="balanced",random_state=RANDOM_STATE),
 "Random Forest":RandomForestClassifier(n_estimators=150,max_features="sqrt",min_samples_leaf=2,
                                         class_weight="balanced_subsample",n_jobs=-1,random_state=RANDOM_STATE)
}
results=[]
fitted={}
for name,clf in classifiers.items():
    pipe=Pipeline([("tfidf",TfidfVectorizer(preprocessor=clean_text,lowercase=False,
        ngram_range=(1,2),min_df=2,max_df=.98,max_features=60000,sublinear_tf=True)),
        ("classifier",clf)])
    print("Fitting",name)
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    pr,rc,f1,_=precision_recall_fscore_support(y_test,pred,average="weighted",zero_division=0)
    results.append({"model":name,"accuracy":accuracy_score(y_test,pred),
                    "precision_weighted":pr,"recall_weighted":rc,"f1_weighted":f1})
    fitted[name]=pipe
results_df=pd.DataFrame(results).sort_values("f1_weighted",ascending=False).reset_index(drop=True)
display(results_df.style.format({c:"{:.3f}" for c in results_df.columns if c!="model"}))
best_name=results_df.iloc[0]["model"]
best_model=fitted[best_name]
print("Selected model by weighted F1:",best_name)

## 5. Model comparison visualization

In [ ]:
ax=results_df.set_index("model")[["accuracy","precision_weighted","recall_weighted","f1_weighted"]].plot(
    kind="bar",figsize=(12,5),ylim=(0,1))
ax.set_ylabel("Score"); ax.set_title("Classifier performance comparison")
plt.xticks(rotation=25,ha="right"); plt.tight_layout(); plt.show()

## 6. Confusion matrices and detailed report

In [ ]:
labels=sorted(df.label.unique())
fig,axes=plt.subplots(2,3,figsize=(16,10))
for ax,(name,pipe) in zip(axes.ravel(),fitted.items()):
    pred=pipe.predict(X_test)
    cm=confusion_matrix(y_test,pred,labels=labels)
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=labels,yticklabels=labels,ax=ax)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()
best_pred=best_model.predict(X_test)
print(classification_report(y_test,best_pred,zero_division=0))

## 7. Optional cross-validation on the training partition

In [ ]:
# This gives a more stable model-selection estimate without using the held-out test set.
# For a full repeatable CV comparison, fit each pipeline on X_train only.
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
cv_rows=[]
for name,clf in classifiers.items():
    pipe=Pipeline([("tfidf",TfidfVectorizer(preprocessor=clean_text,lowercase=False,
        ngram_range=(1,2),min_df=2,max_df=.98,max_features=60000,sublinear_tf=True)),
        ("classifier",clf)])
    scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring=["accuracy","f1_weighted"],n_jobs=1)
    cv_rows.append({"model":name,"cv_accuracy_mean":scores["test_accuracy"].mean(),
                    "cv_f1_weighted_mean":scores["test_f1_weighted"].mean()})
display(pd.DataFrame(cv_rows).sort_values("cv_f1_weighted_mean",ascending=False).round(3))

## 8. Export selected model and results

The project `src/train.py` is the reproducible command-line training entry point and writes the same reports plus `models/best_model.joblib`. To save the notebook-selected model from this runtime:

In [ ]:
import joblib, json
Path("models").mkdir(exist_ok=True); Path("reports").mkdir(exist_ok=True)
joblib.dump(best_model,"models/best_model.joblib")
results_df.to_csv("reports/model_comparison.csv",index=False)
Path("models/model_info.json").write_text(json.dumps({"best_model":best_name,"classes":labels},indent=2))
print("Saved model and comparison report.")